# 1 amending src files

In [1]:
import pandas as pd

def ld(p):
    return pd.read_csv(p, sep=',')

In [2]:
def rm_col(ncols, c):
    if c in ncols:
        ncols.remove(c)
    return ncols

In [3]:
IOU_COLS_GDA = ["3_test_GDA_iou", "3_test/GDA/iou", "1_test/GDA/iou",  
                "2_test/GDA/iou", "1_test_GDA_iou", "2_test_GDA_iou"]
IOU_COLS_SYNT = ["3_test_SYNT_iou", "1_test/SYNT/iou", "2_test/SYNT/iou", 
            "3_test/SYNT/iou", "1_test_SYNT_iou", "2_test_SYNT_iou"]
IOU_COLS_DK = ["3_test_DK_iou", "1_test/DK/iou", "2_test/DK/iou", 
            "3_test/DK/iou", "1_test_DK_iou", "2_test_DK_iou"]

IOU_COLS = IOU_COLS_GDA + IOU_COLS_SYNT + IOU_COLS_DK
# SPLIT_COLUMNS = ['val', 'Sweep']
SCOLS = ["batch_size", "epochs", "warmup_epochs", "wd", "ema", "sub", 
        "ID", "loss", "lrdec", "lrenc", "mod_ph1", "mod_ph2", "Sweep",
        "val", "1_epoch", "2_epoch", "3_epoch"] + IOU_COLS

def ld_scols(p):
    df = pd.read_csv(p, sep=',')
    cols = df.columns
    cols = list(set(cols) & set(SCOLS))
    ncols = list(set(cols) & set(SCOLS))
    df = df[df['State'] == 'finished']
    # if 'ID' in cols:
    #     df.set_index('ID', inplace=True, drop=False)
        # df.reindex('ID')
    if 'ph3' in p:
        ncols = rm_col(ncols, "2_test/DK/iou")
        ncols = rm_col(ncols, "2_test/SYNT/iou")
        ncols = rm_col(ncols, "2_test/GDA/iou")
        ncols = rm_col(ncols, "2_test_DK_iou")
        ncols = rm_col(ncols, "2_test_GDA_iou")
        ncols = rm_col(ncols, "2_test_SYNT_iou")
    if 'S' in p:
        ncols = rm_col(ncols, "1_test_DK_iou")
        ncols = rm_col(ncols, "1_test_GDA_iou")
        ncols = rm_col(ncols, "1_test_SYNT_iou")
    for col in cols:
        if '/' in col:
            # print('sign found in', col)
            df[col.replace('/', '_')] = df[col]
            df = df.drop(columns=[col])
            if col in ncols:
                ncols.remove(col)
            ncols.append(col.replace('/', '_'))
    # print(ncols)
    return df[ncols].fillna({'ema': False})

In [4]:
def map_dict_val(d: dict, val, deff):
    for k, v in d.items():
        if k in val:
            return v
    return deff

In [5]:
fn_tr = {
    'ph2': 'subm',
    'ph3': 'dk',
    'dk': 'dk',
    'subs': 'sub',
    'S': 'sub',
}

def add_tr(df, fn):
    df['train'] = map_dict_val(fn_tr, fn, 's')
    return df

In [6]:
from numpy import select

val_val = {
    "/users/project1/pt01299/synt/gda70/train/index_val.csv": "gda",
    "/users/project1/pt01299/synt/segformer_dataset255_all/val/index.csv": "s",
    "/users/project1/pt01299/synt/mix_val.csv": "m",
    "/users/project1/pt01299/synt/DK/osfstorage/dataset_v2/solardk_dataset_neurips_v2/gentofte_trainval/val/index.csv": "dk"
}

def map_val(df):
    if 'val' not in df.columns:
        df['val'] = list(val_val.keys())[0]
    v = df['val'].astype(str)
    conds = [v.str.contains(k, regex=False, na=False) for k in val_val.keys()]
    choices = list(val_val.values())
    df['val'] = select(conds, choices, default='s')
    return df

In [7]:
def add_tr_val(df):
    df['tr_val'] = df['train'] + '_' + df['val']
    return df

In [8]:
def add_src(df, fn):
    sp = fn.split('_')
    if len(sp) > 2:
        df['src'] = sp[1] + '_' + sp[2].replace('.csv', '')
    elif 'm' in fn:
        df['src'] = 'subm'
    else:
        df['src'] = 'x'
    df['fn'] = fn
    return df

In [9]:
f = 'dk.csv'
f = 'subs_m.csv'
f = 'ph2_dk_dk.csv'
df = ld_scols(f)
df = add_tr(df, f)
df = map_val(df)
df = add_tr_val(df)
df = add_src(df, f)
if 'ID' not in df.columns:
        df['ID'] = 'keine'
df.head()

,lrenc,wd,mod_ph1,ID,ema,epochs,val,lrdec,sub,Sweep,...,warmup_epochs,loss,2_epoch,2_test_SYNT_iou,2_test_DK_iou,2_test_GDA_iou,train,tr_val,src,fn
0,0.000010,0.03,5gyfwlq3,pevtyaco,True,18,gda,0.000050,composite,rbnt81jd,...,2,Dice+Focal,17.0,0.746898,0.808483,0.759344,subm,subm_gda,dk_dk,ph2_dk_dk.csv
1,0.000005,0.01,5gyfwlq3,rkltfhch,False,3,gda,0.000008,mix,rbnt81jd,...,4,Dice+Focal,2.0,0.698779,0.806455,0.758751,subm,subm_gda,dk_dk,ph2_dk_dk.csv
2,0.000005,0.03,lbfcr8ey,uafbq2g8,True,6,gda,0.000008,mix,rbnt81jd,...,4,Dice+Focal,5.0,0.639541,0.813669,0.757724,subm,subm_gda,dk_dk,ph2_dk_dk.csv
3,0.000005,0.05,5gyfwlq3,vymtls23,False,3,s,0.000008,mix,rbnt81jd,...,6,Dice+Focal,2.0,0.705310,0.810658,0.757477,subm,subm_s,dk_dk,ph2_dk_dk.csv
4,0.000005,0.05,lbfcr8ey,usv0usng,True,6,gda,0.000050,mix,rbnt81jd,...,4,Dice+Focal,5.0,0.653817,0.813720,0.757003,subm,subm_gda,dk_dk,ph2_dk_dk.csv


In [10]:
def proc_f(f):
    df = ld_scols(f)
    df = add_tr(df, f)
    df = map_val(df)
    df = add_tr_val(df)
    df = add_src(df, f)
    if 'ID' not in df.columns:
        df['ID'] = 'keine'
        print('no id in', f)
    # df.to_csv(f'modf/{f}')
    
    return df

In [11]:
import os

files = [f for f in os.listdir() if f.lower().endswith(".csv")]
files.remove('da.csv')
files

['30.csv',
 'dk.csv',
 'full.csv',
 'ph2_dk_dk.csv',
 'ph2_dk_gda.csv',
 'ph2_sub_gda.csv',
 'ph2_sub_s.csv',
 'ph2_s_gda.csv',
 'ph2_s_s.csv',
 'ph3_subm_gda.csv',
 'ph3_subm_m.csv',
 'ph3_subm_s.csv',
 'S3.csv',
 'S4.csv',
 'short.csv',
 'subs_16.csv',
 'subs_16A.csv',
 'subs_m.csv']

## proces each file

In [12]:
ph1 = pd.DataFrame()
ph2 = pd.DataFrame()
ph3 = pd.DataFrame()

for ff in files:
    print(ff)
    df = proc_f(ff)
    if 'ph2' in ff:
        ph2 = pd.concat([df, ph2], ignore_index=True)
    elif 'ph3' in ff:
        ph3 = pd.concat([df, ph3], ignore_index=True)
    else:
        try:
            if 'S' in ff:
                print(len(df.columns))
                print(len(pd.unique(df.columns)))
                print(df.columns)
            ph1 = pd.concat([df, ph1], ignore_index=True)
        except Exception as e:
            print('ph1', ff, e)

# ph1 = ph1.dropna(axis='columns')
ph2 = ph2.dropna(axis='columns')
ph3 = ph3.dropna(axis='columns')

ph1.to_csv('modf/proc_ph1.csv')
ph2.to_csv('modf/proc_ph2.csv')
ph3.to_csv('modf/proc_ph3.csv')

30.csv
dk.csv
full.csv
ph2_dk_dk.csv
ph2_dk_gda.csv
ph2_sub_gda.csv
ph2_sub_s.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_3756\3901077901.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})
C:\Users\admin\AppData\Local\Temp\ipykernel_3756\3901077901.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})


ph2_s_gda.csv
ph2_s_s.csv
ph3_subm_gda.csv
ph3_subm_m.csv
ph3_subm_s.csv
S3.csv
19
19
Index(['lrenc', 'wd', 'val', 'ID', 'epochs', 'lrdec', 'sub', 'Sweep',
       'batch_size', 'warmup_epochs', '1_epoch', 'loss', '1_test_DK_iou',
       '1_test_SYNT_iou', '1_test_GDA_iou', 'train', 'tr_val', 'src', 'fn'],
      dtype='object')
S4.csv
17
17
Index(['lrenc', 'wd', 'ID', 'lrdec', 'sub', 'batch_size', 'warmup_epochs',
       '1_epoch', 'loss', '1_test_SYNT_iou', '1_test_DK_iou', '1_test_GDA_iou',
       'train', 'val', 'tr_val', 'src', 'fn'],
      dtype='object')
short.csv
subs_16.csv
subs_16A.csv
subs_m.csv
no id in subs_m.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_3756\3901077901.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})
C:\Users\admin\AppData\Local\Temp\ipykernel_3756\3901077901.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})


drop duplicate iou cols
drop empty ious
====stack all
====join 1-2 by mod1
====join 2-3 by mod3
====generate comb_key 1-2-3
====grouped agg stats by comb_key
stack disjoint to joint - to załatwia skrypt etl, tutaj tylko prawdziwe i pełne fazy

# 2 joins + save wide

In [13]:
cnc = pd.concat([ph1, ph2, ph3])
cnc.head()

,lrenc,wd,mod_ph1,ema,val,epochs,lrdec,sub,Sweep,batch_size,...,ID,2_epoch,2_test_SYNT_iou,2_test_DK_iou,2_test_GDA_iou,mod_ph2,3_epoch,3_test_DK_iou,3_test_SYNT_iou,3_test_GDA_iou
0,0.00001,0.05,NaN,True,m,20.0,0.000008,composite,8ej2yv2y,8,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.00001,0.01,NaN,False,s,20.0,0.000008,15,8ej2yv2y,8,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.00001,0.03,NaN,True,m,20.0,0.000008,15,8ej2yv2y,16,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.00001,0.03,NaN,False,gda,20.0,0.000008,25,8ej2yv2y,16,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.00001,0.05,NaN,False,gda,20.0,0.000008,15,8ej2yv2y,16,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
cnc.to_csv('modf/cnc123b.csv')

In [467]:
# df12 = pd.merge(ph1, ph2, how='inner', left_on=['tr_val'], right_on=['src'])
df12 = pd.merge(ph1, ph2, how='inner', left_on=['ID'], right_on=['mod_ph1'])
df12.head()

,val_x,lrenc_x,mod_ph1_x,epochs_x,lrdec_x,loss_x,batch_size_x,1_epoch,wd_x,warmup_epochs_x,...,ema_y,sub_y,2_epoch,2_test_DK_iou,2_test_SYNT_iou,2_test_GDA_iou,train_y,tr_val_y,src_y,fn_y
0,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,16,15.0,0.05,6,...,False,15,2.0,0.789495,0.788783,0.739968,subm,subm_gda,sub_gda,ph2_sub_gda.csv
1,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,16,15.0,0.05,6,...,False,composite,5.0,0.808589,0.789358,0.736804,subm,subm_s,sub_gda,ph2_sub_gda.csv
2,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,16,15.0,0.05,6,...,False,15,2.0,0.788369,0.791244,0.736472,subm,subm_gda,sub_gda,ph2_sub_gda.csv
3,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,16,15.0,0.05,6,...,False,mix,5.0,0.801673,0.783520,0.734996,subm,subm_m,sub_gda,ph2_sub_gda.csv
4,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,16,15.0,0.05,6,...,False,15,2.0,0.803570,0.799280,0.734078,subm,subm_gda,sub_gda,ph2_sub_gda.csv


In [468]:
df123 = pd.merge(df12, ph3, how='inner', left_on=['ID_y'], right_on=['mod_ph2'])
df123.head()

,val_x,lrenc_x,mod_ph1_x,epochs_x,lrdec_x,loss_x,batch_size_x,1_epoch,wd_x,warmup_epochs_x,...,wd,loss,Sweep,3_test_GDA_iou,3_test_SYNT_iou,3_test_DK_iou,train,tr_val,src,fn
0,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,0.03,Dice+Focal,r4oohhlg,0.750134,0.792205,0.811760,dk,dk_gda,subm_m,ph3_subm_m.csv
1,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,0.05,Dice+Focal,r4oohhlg,0.748709,0.803058,0.813642,dk,dk_gda,subm_m,ph3_subm_m.csv
2,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,0.01,Dice+Focal,r4oohhlg,0.748688,0.803828,0.814052,dk,dk_dk,subm_m,ph3_subm_m.csv
3,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,0.01,Dice+Focal,r4oohhlg,0.748615,0.804047,0.813060,dk,dk_gda,subm_m,ph3_subm_m.csv
4,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,0.03,Dice+Focal,r4oohhlg,0.748261,0.802384,0.813662,dk,dk_gda,subm_m,ph3_subm_m.csv


In [ ]:
df = df123

# reduntant but visible join sides
# df['comb_key'] = df['src_x'] + '_' + df['tr_val_x'] + '|' + \
#                 df['src_y'] + '_' + df['tr_val_y'] + '|' + \
#                 df['src'] + '_' + df['tr_val']

df['comb_key'] = df['tr_val_x'] + '|' + \
                df['tr_val_y'] + '|' + \
                df['tr_val']
df.head()

,val_x,lrenc_x,mod_ph1_x,epochs_x,lrdec_x,loss_x,batch_size_x,1_epoch,wd_x,warmup_epochs_x,...,loss,Sweep,3_test_GDA_iou,3_test_SYNT_iou,3_test_DK_iou,train,tr_val,src,fn,comb_key
0,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,Dice+Focal,r4oohhlg,0.750134,0.792205,0.811760,dk,dk_gda,subm_m,ph3_subm_m.csv,sub_gda|subm_m|dk_gda
1,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,Dice+Focal,r4oohhlg,0.748709,0.803058,0.813642,dk,dk_gda,subm_m,ph3_subm_m.csv,sub_gda|subm_m|dk_gda
2,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,Dice+Focal,r4oohhlg,0.748688,0.803828,0.814052,dk,dk_dk,subm_m,ph3_subm_m.csv,sub_gda|subm_m|dk_dk
3,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,Dice+Focal,r4oohhlg,0.748615,0.804047,0.813060,dk,dk_gda,subm_m,ph3_subm_m.csv,sub_gda|subm_m|dk_gda
4,gda,0.00005,NaN,16.0,0.000008,Dice+Focal,8,15.0,0.05,6,...,Dice+Focal,r4oohhlg,0.748261,0.802384,0.813662,dk,dk_gda,subm_m,ph3_subm_m.csv,sub_gda|subm_m|dk_gda


In [ ]:
# df.to_csv('modf/ph123b.csv') # OK

# 2b not used

In [471]:
def compute_univariate_stats(series):
    numeric = pd.to_numeric(series, errors="coerce")
    desc = numeric.describe(percentiles=[0.25, 0.5, 0.75])
    # stats_map = {
    #     "count": desc.get("count", np.nan),
    #     "mean": desc.get("mean", np.nan),
    #     "std": desc.get("std", np.nan),
    #     "min": desc.get("min", np.nan),
    #     "25%": desc.get("25%", np.nan),
    #     "50%": desc.get("50%", np.nan),
    #     "75%": desc.get("75%", np.nan),
    #     "max": desc.get("max", np.nan),
    # }
    # return pd.Series(stats_map)
    return desc

In [472]:
# dfst = compute_univariate_stats(df)
# dfst
dsc = df.describe(percentiles=[0.25, 0.5, 0.75]) #['epochs_x']

cols = dsc.columns
cols = list(set(cols) & set(IOU_COLS))

# dsc[cols].dropna(axis=1)
dsc = dsc[cols]
dsc

,2_test_GDA_iou,3_test_SYNT_iou,3_test_DK_iou,3_test_GDA_iou,1_test_GDA_iou,1_test_SYNT_iou,1_test_DK_iou,2_test_SYNT_iou,2_test_DK_iou
count,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000
mean,0.746488,0.804743,0.806759,0.745333,0.699929,0.776307,0.637260,0.810371,0.800297
std,0.002451,0.020219,0.007988,0.003918,0.010267,0.055562,0.075546,0.019053,0.009148
min,0.742180,0.724750,0.755980,0.730904,0.662182,0.629262,0.504851,0.750751,0.754951
25%,0.744672,0.794626,0.802214,0.743049,0.695381,0.750989,0.585099,0.800917,0.795903
50%,0.746855,0.804057,0.807269,0.745485,0.700820,0.789735,0.602719,0.811116,0.801553
75%,0.748329,0.811966,0.812249,0.747980,0.704403,0.810782,0.728871,0.817816,0.805206
max,0.752292,0.844469,0.823754,0.757549,0.720929,0.837420,0.774644,0.844630,0.817603


# 3 not used

In [473]:
# df_x = df[['comb_key', 'trg_var', 'ph_nr_x', 'stat_mean_x']].copy()
# df_x.columns = ['comb_key', 'trg_var', 'ph_nr', 'stat_mean']

# df_y = df[['comb_key', 'trg_var', 'ph_nr_y', 'stat_mean_y']].copy()
# df_y.columns = ['comb_key', 'trg_var', 'ph_nr', 'stat_mean']

# df_blank = df[['comb_key', 'trg_var', 'ph_nr', 'stat_mean']].copy()

# # Combine (stack) them all together into one long format dataframe
# df_long = pd.concat([df_x, df_y, df_blank], ignore_index=True)

In [487]:
# dfst = dfst.describe(percentiles=[0.25, 0.5, 0.75])
cols = df.columns
cols = list(set(cols) & set(IOU_COLS)) + ['comb_key']
dfst = df[cols].groupby(['comb_key']).agg(['mean', 'max'])
dfst

2_test_GDA_iou           3_test_SYNT_iou            \
                                  mean       max            mean       max   
comb_key                                                                     
s_gda|subm_gda|dk_dk          0.746564  0.747124        0.805660  0.836882   
s_gda|subm_gda|dk_gda         0.746601  0.747124        0.808574  0.836495   
s_gda|subm_m|dk_dk            0.747463  0.749387        0.812605  0.835097   
s_gda|subm_m|dk_gda           0.747188  0.749387        0.813242  0.834807   
s_gda|subm_s|dk_dk            0.743916  0.747182        0.805593  0.841025   
s_gda|subm_s|dk_gda           0.743961  0.747182        0.812244  0.841026   
s_s|subm_gda|dk_dk            0.748922  0.749163        0.804856  0.812389   
s_s|subm_gda|dk_gda           0.748736  0.749163        0.802817  0.809316   
s_s|subm_m|dk_dk              0.746846  0.746855        0.804388  0.809798   
s_s|subm_m|dk_gda             0.746843  0.746855        0.804909  0.810073   
s_s|subm_s|dk_dk              0.748952  0.752292        0.796472  0.811137   
s_s|subm_s|dk_gda             0.748416  0.752292        0.795989  0.810350   
sub_gda|subm_gda|dk_dk        0.744252  0.746150        0.812204  0.823121   
sub_gda|subm_gda|dk_gda       0.743800  0.746150        0.811702  0.824424   
sub_gda|subm_m|dk_dk          0.745954  0.747984        0.801622  0.810519   
sub_gda|subm_m|dk_gda         0.745989  0.747984        0.802948  0.811636   
sub_gda|subm_s|dk_dk          0.745546  0.748329        0.799696  0.817168   
sub_gda|subm_s|dk_gda         0.745888  0.748329        0.801339  0.816936   
sub_s|subm_gda|dk_dk          0.746770  0.749760        0.775363  0.803113   
sub_s|subm_gda|dk_gda         0.746928  0.749760        0.773112  0.804078   
sub_s|subm_m|dk_dk            0.745421  0.747667        0.810187  0.838262   
sub_s|subm_m|dk_gda           0.745380  0.747667        0.809854  0.837269   
sub_s|subm_s|dk_dk            0.743866  0.744896        0.824158  0.844469   
sub_s|subm_s|dk_gda           0.743858  0.744896        0.823127  0.842732   

                        3_test_DK_iou           3_test_GDA_iou            \
                                 mean       max           mean       max   
comb_key                                                                   
s_gda|subm_gda|dk_dk         0.806470  0.818699       0.744667  0.750513   
s_gda|subm_gda|dk_gda        0.806060  0.817590       0.745613  0.748235   
s_gda|subm_m|dk_dk           0.804464  0.816450       0.747558  0.751661   
s_gda|subm_m|dk_gda          0.805771  0.815439       0.747062  0.751476   
s_gda|subm_s|dk_dk           0.807309  0.815770       0.742067  0.751120   
s_gda|subm_s|dk_gda          0.804663  0.816414       0.742711  0.749432   
s_s|subm_gda|dk_dk           0.810356  0.817388       0.746413  0.749239   
s_s|subm_gda|dk_gda          0.810854  0.816640       0.745366  0.749092   
s_s|subm_m|dk_dk             0.814410  0.817758       0.746140  0.747766   
s_s|subm_m|dk_gda            0.811653  0.818361       0.746148  0.751909   
s_s|subm_s|dk_dk             0.808729  0.820155       0.747646  0.753790   
s_s|subm_s|dk_gda            0.806434  0.819292       0.747267  0.753114   
sub_gda|subm_gda|dk_dk       0.813440  0.820554       0.742957  0.746624   
sub_gda|subm_gda|dk_gda      0.813787  0.822836       0.741897  0.746718   
sub_gda|subm_m|dk_dk         0.807829  0.814652       0.745107  0.748688   
sub_gda|subm_m|dk_gda        0.807914  0.816086       0.744881  0.751804   
sub_gda|subm_s|dk_dk         0.803799  0.817498       0.743341  0.751754   
sub_gda|subm_s|dk_gda        0.807125  0.817405       0.743420  0.748682   
sub_s|subm_gda|dk_dk         0.806723  0.820677       0.746689  0.754875   
sub_s|subm_gda|dk_gda        0.808195  0.822673       0.747548  0.757549   
sub_s|subm_m|dk_dk           0.800477  0.821736       0.743810  0.748784   
sub_s|subm_m|dk_gda          0.806619  0.823754       0.741709  0.747507   
sub_s|subm_s|dk_d

In [ ]:
dfst = df.groupby(['comb_key']).agg(mean1GDA=('1_test_GDA_iou', 'mean'), 
                                  mean2GDA=('2_test_GDA_iou', 'mean'), 
                                  mean3GDA=('3_test_GDA_iou', 'mean'),
                                  max1GDA=('1_test_GDA_iou', 'max'), 
                                  max2GDA=('2_test_GDA_iou', 'max'), 
                                  max3GDA=('3_test_GDA_iou', 'max'),
                                  mean1DK=('1_test_DK_iou', 'mean'), 
                                  mean2DK=('2_test_DK_iou', 'mean'), 
                                  mean3DK=('3_test_DK_iou', 'mean'),
                                  max1DK=('1_test_DK_iou', 'max'), 
                                  max2DK=('2_test_DK_iou', 'max'), 
                                  max3DK=('3_test_DK_iou', 'max'),
                                  mean1SYNT=('1_test_SYNT_iou', 'mean'), 
                                  mean2SYNT=('2_test_SYNT_iou', 'mean'), 
                                  mean3SYNT=('3_test_SYNT_iou', 'mean'),
                                  max1SYNT=('1_test_SYNT_iou', 'max'), 
                                  max2SYNT=('2_test_SYNT_iou', 'max'), 
                                  max3SYNT=('3_test_SYNT_iou', 'max'),
                                  ).reset_index()
dfst

,comb_key,mean1GDA,mean2GDA,mean3GDA,max1GDA,max2GDA,max3GDA
0,s_gda|subm_gda|dk_dk,0.681078,0.746564,0.744667,0.702050,0.747124,0.750513
1,s_gda|subm_gda|dk_gda,0.678647,0.746601,0.745613,0.702050,0.747124,0.748235
2,s_gda|subm_m|dk_dk,0.697551,0.747463,0.747558,0.704403,0.749387,0.751661
3,s_gda|subm_m|dk_gda,0.699410,0.747188,0.747062,0.704403,0.749387,0.751476
4,s_gda|subm_s|dk_dk,0.698822,0.743916,0.742067,0.704247,0.747182,0.751120
5,s_gda|subm_s|dk_gda,0.700571,0.743961,0.742711,0.704247,0.747182,0.749432
6,s_s|subm_gda|dk_dk,0.700820,0.748922,0.746413,0.700820,0.749163,0.749239
7,s_s|subm_gda|dk_gda,0.700820,0.748736,0.745366,0.700820,0.749163,0.749092
8,s_s|subm_m|dk_dk,0.702245,0.746846,0.746140,0.702389,0.746855,0.747766
9,s_s|subm_m|dk_gda,0.702206,0.746843,0.746148,0.702389,0.746855,0.751909


In [479]:
d1 = dfst[['comb_key', 'mean1GDA', 'max1GDA']].copy().rename(columns={'mean1GDA': 'meanGDA', 'max1GDA': 'maxGDA'})
d1['ph'] = 1

d2 = dfst[['comb_key', 'mean2GDA', 'max2GDA']].copy().rename(columns={'mean2GDA': 'meanGDA', 'max2GDA': 'maxGDA'})
d2['ph'] = 2

d3 = dfst[['comb_key', 'mean3GDA', 'max3GDA']].copy().rename(columns={'mean3GDA': 'meanGDA', 'max3GDA': 'maxGDA'})
d3['ph'] = 3

df_long = pd.concat([d1, d2, d3], ignore_index=True)
df_long

,comb_key,meanGDA,maxGDA,ph
0,s_gda|subm_gda|dk_dk,0.681078,0.702050,1
1,s_gda|subm_gda|dk_gda,0.678647,0.702050,1
2,s_gda|subm_m|dk_dk,0.697551,0.704403,1
3,s_gda|subm_m|dk_gda,0.699410,0.704403,1
4,s_gda|subm_s|dk_dk,0.698822,0.704247,1
...,...,...,...,...
67,sub_s|subm_gda|dk_gda,0.747548,0.757549,3
68,sub_s|subm_m|dk_dk,0.743810,0.748784,3
69,sub_s|subm_m|dk_gda,0.741709,0.747507,3
70,sub_s|subm_s|dk_dk,0.742711,0.746902,3


In [ ]:
# df_long.to_csv('modf/long.csv')